# PALMAS Frame Extraction Pipeline — ALL CLASSES (corrected, consistent)

Unified, corrected replacement for the previous Palmas extraction notebook(s).
Fixes the resolution inconsistency found across classes by running **every class through
the exact same code path**: same frame-sampling method, same resize step, same safe
Drive-copy pattern — so Normal / Leve / Moderada can no longer silently diverge.

What changed vs. the older notebook(s):
- **Resize to 224x224 is now unconditional** — applied to every saved frame,
  for every class. This was previously missing for one class, which is exactly what caused the
  earlier mismatch (some subjects ended up at raw capture resolution instead of the target size).
- **One discard method for all classes**: 25% of frames discarded at the start, 10% at the end, by percentage of that video's total frame count (matches the method already used for all three Palmas classes). No per-class special-casing.
- **Safe Drive I/O**: each video is copied locally, processed, copied back, and the file count on
  Drive is verified against the local output *before* moving on — the same pattern that already
  works reliably elsewhere in this project. This catches silent Drive-sync data loss immediately
  instead of discovering it later.
- **Built-in resolution self-check** (final section): after processing, every saved frame for every
  subject is checked against the target size and any mismatch is written to a log file instead of
  being discovered by hand later.

Expected folder structure:
```
Dataset/
└── palmas/{Leve,Moderada,Normal}/*.mp4
```

You can re-run this on all three classes safely — subjects that are already correct
(already 224x224, already the right frame count) will simply be re-written
identically. If you only want to fix the affected class, set `CLASSES` below to just that one class.


## 1. Mount Drive & install dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%pip install opencv-python tqdm numpy --quiet

## 2. Config
Edit `PALMAS_DIR` and `OUTPUT_ROOT` to match your Drive paths. `CLASSES` controls which class folders get (re-)processed — leave all three in for a full, consistent run, or trim it down to just the class you need to fix.

In [9]:
import shutil
from pathlib import Path
from tqdm import tqdm
import cv2
import numpy as np

# ---------------------------------------------------------------------------
# CONFIG — edit these to match your setup
# ---------------------------------------------------------------------------
PALMAS_DIR   = Path('/content/drive/MyDrive/Dataset/palmas')   # folder with Leve/Moderada/Normal subfolders
OUTPUT_ROOT = Path('/content/drive/MyDrive/dataset_palmas_frames')                     # where palmas_frames/ will be written

CLASSES = ['Leve', 'Moderada', 'Normal']   # run all three for a fully consistent dataset
VIDEO_EXTS = ['.mp4', '.mov', '.avi', '.MP4', '.MOV', '.AVI']

PALMAS_DISCARD_START_PCT = 0.25    # discard this fraction of frames at the start of each clip
PALMAS_DISCARD_END_PCT   = 0.1    # discard this fraction of frames at the end of each clip
PALMAS_NUM_FRAMES        = 16      # number of evenly-spaced frames to extract per video
IMG_SIZE = (224, 224)       # every saved frame is resized to this, no exceptions

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


## 3. `get_frame_indices` / `process_video`
Discards the configured start/end fraction of the clip, then samples `PALMAS_NUM_FRAMES` evenly-spaced frames from what's left. Every frame is resized to `IMG_SIZE` **before** being written to disk — this is the step that was missing for one class previously, so it is deliberately unconditional here (no per-class branch, no way to skip it by accident).

Returns `(frames_saved, error)` — `error` is `None` on success, otherwise a short string describing what went wrong.

In [10]:
def get_frame_indices(total_frames, discard_start_pct, discard_end_pct, num_samples):
    """Return `num_samples` frame indices, uniformly spaced, after discarding
    the given fraction of frames from the start and end of the clip."""
    start = int(total_frames * discard_start_pct)
    end = int(total_frames * (1 - discard_end_pct))

    if end <= start:
        # Degenerate case (very short clip) -> fall back to the full video
        start, end = 0, total_frames

    if total_frames < num_samples:
        return list(range(total_frames))

    indices = np.linspace(start, end - 1, num_samples)
    indices = np.round(indices).astype(int)
    indices = np.clip(indices, 0, total_frames - 1)
    return indices.tolist()


def process_video(video_path, out_dir, discard_start_pct, discard_end_pct, num_frames, size=IMG_SIZE):
    video_path = Path(video_path)
    out_dir = Path(out_dir)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return 0, 'could_not_open_video'

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        return 0, 'zero_or_unreadable_frame_count'

    frame_indices = get_frame_indices(total_frames, discard_start_pct, discard_end_pct, num_frames)

    out_dir.mkdir(parents=True, exist_ok=True)

    saved = 0
    for i, frame_idx in enumerate(frame_indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        ok, frame = cap.read()
        if not ok:
            continue
        # Resize is unconditional -- every frame, every class, no exceptions.
        frame = cv2.resize(frame, size, interpolation=cv2.INTER_AREA)
        out_path = out_dir / f'frame_{i:03d}.jpg'
        cv2.imwrite(str(out_path), frame)
        saved += 1

    cap.release()

    if saved == 0:
        return 0, 'no_frames_could_be_read'

    return saved, None


## 4. Logging containers

In [ ]:
corrupted_log = []
summary_rows = []

## 5. Run the pipeline
Same safe pattern used elsewhere in this project: copy one video to local disk, process it locally, copy the finished frame folder back to Drive, **verify the file count actually landed on Drive**, then delete the local copies — one video at a time, so local disk never fills up and a Drive-sync failure is caught immediately instead of silently.

In [ ]:
LOCAL_TMP = Path('/content/palmas_tmp')
LOCAL_TMP.mkdir(parents=True, exist_ok=True)

print("Processing PALMAS videos...")
for cls in CLASSES:
    cls_dir = PALMAS_DIR / cls
    if not cls_dir.exists():
        print(f"  [WARN] missing class folder: {cls_dir}")
        continue

    video_files = sorted([p for p in cls_dir.iterdir() if p.suffix in VIDEO_EXTS])
    for video_path in tqdm(video_files, desc=f"palmas/{cls}"):
        subject_id = video_path.stem

        # 1. Copy ONE video to local disk
        local_video = LOCAL_TMP / video_path.name
        shutil.copy2(video_path, local_video)

        # 2. Process it -> write frames to a LOCAL folder first (not Drive)
        local_out_dir = LOCAL_TMP / 'out' / cls / subject_id
        saved, error = process_video(
            local_video, local_out_dir,
            PALMAS_DISCARD_START_PCT, PALMAS_DISCARD_END_PCT, PALMAS_NUM_FRAMES
        )

        out_dir = OUTPUT_ROOT / 'palmas_frames' / cls / subject_id
        if error:
            corrupted_log.append({
                'modality': 'palmas', 'class': cls, 'subject_id': subject_id,
                'file': str(video_path), 'error': error
            })
        else:
            # 3. Copy the finished folder back to Drive, ONE folder at a time
            shutil.copytree(local_out_dir, out_dir, dirs_exist_ok=True)

            # 4. Verify the copy actually landed on Drive before trusting it
            n_local = len(list(local_out_dir.glob('*')))
            n_drive = len(list(out_dir.glob('*')))
            if n_local != n_drive:
                corrupted_log.append({
                    'modality': 'palmas', 'class': cls, 'subject_id': subject_id,
                    'file': str(video_path), 'error': f'drive_copy_mismatch_{n_local}_vs_{n_drive}'
                })

        summary_rows.append({
            'modality': 'palmas', 'class': cls, 'subject_id': subject_id,
            'frames_saved_or_0': saved
        })

        # 5. Clean up local copies so /content doesn't fill up over hundreds of videos
        local_video.unlink(missing_ok=True)
        shutil.rmtree(local_out_dir, ignore_errors=True)

print("PALMAS done.")


Processing PALMAS videos...


palmas/Normal:  19%|█▉        | 105/556 [16:09<1:09:53,  9.30s/it]

In [15]:
summary_rows = []
corrupted_log = []

In [16]:
from pathlib import Path
from PIL import Image
import shutil

LOCAL_TMP = Path('/content/palmas_tmp')
LOCAL_TMP.mkdir(parents=True, exist_ok=True)


def is_already_done(out_dir, expected_frames, expected_size):
    if not out_dir.exists():
        return False

    frame_files = sorted(out_dir.glob('frame_*.jpg'))

    if len(frame_files) != expected_frames:
        return False

    try:
        with Image.open(frame_files[0]) as im:
            return im.size == expected_size
    except Exception:
        return False


print("Processing PALMAS Normal videos...")

# ONLY process Normal
for cls in ['Normal']:

    cls_dir = PALMAS_DIR / cls

    if not cls_dir.exists():
        print(f"  [WARN] missing class folder: {cls_dir}")
        continue

    video_files = sorted([
        p for p in cls_dir.iterdir()
        if p.suffix in VIDEO_EXTS
    ])

    for video_path in tqdm(video_files, desc=f"palmas/{cls}"):

        subject_id = video_path.stem

        out_dir = (
            OUTPUT_ROOT /
            'palmas_frames' /
            cls /
            subject_id
        )

        # Skip subjects that were already successfully processed
        if is_already_done(
            out_dir,
            PALMAS_NUM_FRAMES,
            IMG_SIZE
        ):
            continue

        local_video = LOCAL_TMP / video_path.name
        local_out_dir = (
            LOCAL_TMP /
            'out' /
            cls /
            subject_id
        )

        saved = 0

        try:

            # 1. Drive -> local
            shutil.copy2(
                video_path,
                local_video
            )

            # 2. Process locally
            saved, error = process_video(
                local_video,
                local_out_dir,
                PALMAS_DISCARD_START_PCT,
                PALMAS_DISCARD_END_PCT,
                PALMAS_NUM_FRAMES
            )

            if error:

                corrupted_log.append({
                    'modality': 'palmas',
                    'class': cls,
                    'subject_id': subject_id,
                    'file': str(video_path),
                    'error': error
                })

            else:

                # 3. Local -> Drive
                shutil.copytree(
                    local_out_dir,
                    out_dir,
                    dirs_exist_ok=True
                )

                # 4. Verify copy
                n_local = len(
                    list(local_out_dir.glob('*'))
                )

                n_drive = len(
                    list(out_dir.glob('*'))
                )

                if n_local != n_drive:

                    corrupted_log.append({
                        'modality': 'palmas',
                        'class': cls,
                        'subject_id': subject_id,
                        'file': str(video_path),
                        'error': (
                            f'drive_copy_mismatch_'
                            f'{n_local}_vs_{n_drive}'
                        )
                    })

            summary_rows.append({
                'modality': 'palmas',
                'class': cls,
                'subject_id': subject_id,
                'frames_saved_or_0': saved
            })

        except OSError as e:

            corrupted_log.append({
                'modality': 'palmas',
                'class': cls,
                'subject_id': subject_id,
                'file': str(video_path),
                'error': f'io_error:{e}'
            })

            print(
                f"  [WARN] I/O error on {subject_id}, "
                f"will retry on next run: {e}"
            )

        finally:

            # Always clean local files
            local_video.unlink(
                missing_ok=True
            )

            shutil.rmtree(
                local_out_dir,
                ignore_errors=True
            )


print("PALMAS Normal done.")

Processing PALMAS Normal videos...


palmas/Normal: 100%|██████████| 556/556 [1:23:55<00:00,  9.06s/it]

PALMAS Normal done.


## 6. Save logs
Writes the summary and any corrupted/mismatched entries to CSV on Drive.

In [17]:
import csv

summary_csv = OUTPUT_ROOT / 'palmas_frame_summary.csv'
with open(summary_csv, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['modality', 'class', 'subject_id', 'frames_saved_or_0'])
    writer.writeheader()
    writer.writerows(summary_rows)
print(f"Summary saved: {summary_csv} ({len(summary_rows)} rows)")

if corrupted_log:
    corrupted_csv = OUTPUT_ROOT / 'palmas_corrupted_log.csv'
    with open(corrupted_csv, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['modality', 'class', 'subject_id', 'file', 'error'])
        writer.writeheader()
        writer.writerows(corrupted_log)
    print(f"[WARN] {len(corrupted_log)} problem videos logged: {corrupted_csv}")
else:
    print("No corrupted or mismatched videos.")


Summary saved: /content/drive/MyDrive/dataset_palmas_frames/palmas_frame_summary.csv (554 rows)
No corrupted or mismatched videos.


## 7. Resolution self-check (do this every time)
Reads back every saved frame for every subject and confirms it is exactly `224x224` and that each subject has exactly `PALMAS_NUM_FRAMES` frames. This is exactly the check that would have caught the earlier Yemas Leve/Normal/Moderada mismatch immediately instead of it being found later by hand — run this before trusting any export of this folder.

In [18]:
from PIL import Image

resolution_mismatch_log = []
expected_size = IMG_SIZE
expected_count = PALMAS_NUM_FRAMES

for cls in CLASSES:
    cls_out_dir = OUTPUT_ROOT / 'palmas_frames' / cls
    if not cls_out_dir.exists():
        continue
    for subject_dir in sorted(cls_out_dir.iterdir()):
        if not subject_dir.is_dir():
            continue
        frame_files = sorted(subject_dir.glob('frame_*.jpg'))
        if len(frame_files) != expected_count:
            resolution_mismatch_log.append({
                'class': cls, 'subject_id': subject_dir.name,
                'issue': f'frame_count_{len(frame_files)}_expected_{expected_count}'
            })
            continue
        # Check the first frame only, per subject, to keep this fast; every frame in a
        # subject folder comes from the same process_video() call so this is representative.
        with Image.open(frame_files[0]) as im:
            if im.size != expected_size:
                resolution_mismatch_log.append({
                    'class': cls, 'subject_id': subject_dir.name,
                    'issue': f'size_{im.size[0]}x{im.size[1]}_expected_{expected_size[0]}x{expected_size[1]}'
                })

if resolution_mismatch_log:
    mismatch_csv = OUTPUT_ROOT / 'palmas_resolution_mismatch_log.csv'
    import csv
    with open(mismatch_csv, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['class', 'subject_id', 'issue'])
        writer.writeheader()
        writer.writerows(resolution_mismatch_log)
    print(f"[FAIL] {len(resolution_mismatch_log)} subject(s) do not match the expected size/frame count.")
    print(f"       Logged to {mismatch_csv} -- do not treat this export as final until this is empty.")
else:
    print(f"[PASS] Every subject in {CLASSES} has exactly {expected_count} frames at {expected_size[0]}x{expected_size[1]}.")


[PASS] Every subject in ['Leve', 'Moderada', 'Normal'] has exactly 16 frames at 224x224.
